# 01 — Data Quality Audit

**Business purpose.** Establish whether the source can support reliable aggregation before interpreting commercial metrics. This notebook does not auto-delete unusual values. Run `python -m src.ingest --inspect-only`, then ingestion and audit first.

**Source:** UCI Online Retail II. This notebook is for the real source only; synthetic records belong exclusively in unit tests.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from src.config import PROJECT_ROOT, DEFAULT_INGESTED, PROCESSED_DIR
from src.validate import audit_dataframe

print("Project root:", PROJECT_ROOT)
if not DEFAULT_INGESTED.exists():
    raise FileNotFoundError("Run `python -m src.ingest` after placing the official workbook at data/raw/online_retail_II.xlsx.")
raw = pd.read_csv(DEFAULT_INGESTED, low_memory=False, dtype={"InvoiceNo":"string", "StockCode":"string", "CustomerID":"string"})
print("Rows:", f"{len(raw):,}")
print("Columns:", list(raw.columns))
raw.head()

## Audit metrics

Review both the affected row count and percentage. The treatment column is part of the audit trail; a large count is not, by itself, a reason to delete those rows.

In [ ]:
audit, audit_detail = audit_dataframe(raw)
display(audit)
audit.to_csv(PROCESSED_DIR / "notebook_data_quality_audit.csv", index=False)
print("Invoice/line-value rule:", audit_detail["calculation_notes"])

## Inspect invoice-level inconsistency

Different dates, CustomerIDs, or countries attached to one invoice can distort invoice/customer metrics if a single representative is silently chosen.

In [ ]:
invoice = raw.copy()
invoice["InvoiceDate_parsed"] = pd.to_datetime(invoice["InvoiceDate"], errors="coerce", format="mixed")
check_cols = [c for c in ["InvoiceDate_parsed", "CustomerID", "Country"] if c in invoice.columns]
variation = invoice.dropna(subset=["InvoiceNo"]).groupby("InvoiceNo")[check_cols].nunique(dropna=True)
variant_counts = (variation > 1).sum()
variation_summary = pd.DataFrame({
    "invoices_with_variants": variant_counts,
    "invoice_count": len(variation),
})
variation_summary.index.name = "invoice_field"
display(variation_summary)
display(variation.loc[(variation > 1).any(axis=1)].head(20))

## Interpretation checkpoint

Write your own three-sentence interpretation after running the real dataset: (1) which issues are materially common, (2) which metrics they limit, and (3) what was preserved rather than deleted. Do not claim that a specific issue is common until the table above supports it.